# Resource estimation
$
\newcommand{\ket}[1]{\left|#1\right\rangle}
\newcommand{\bra}[1]{\left\langle #1\right|}
\newcommand{\braket}[2]{\left\langle #1 \middle| #2 \right\rangle}
\newcommand{\ketbra}[2]{\left|#1\right\rangle\!\left\langle #2\right|}
$


In [1]:
# allows us to have visibility on our package without installing it in editing mode
import sys;
if ".." not in sys.path: sys.path.append("..")

import itertools
import numpy as np
from qiskit.circuit import QuantumCircuit, Gate
from qiskit.converters import circuit_to_dag
from monaqa2.qiskit.utils_numpy import kron, ketbra, bra, ket
from monaqa2.qiskit.utils_qiskit import get_unitary
import sympy as sp
from sympy import latex
from IPython.display import display, Markdown

def lx(expr):
    return latex(expr).replace('*', '')

This notebook details the resource needed to implement the walk operator


$$
W = R_0 V^\dagger B^\dagger F B V.
$$

where

* $ V\ket{x}_a\ket{0}_b = \ket{x}_a \sum_y \sqrt{P_{yx}}\ket{y}_b $ is the move proposal unitary and the proposal here can be the uniform move, the local spin flip (single or multiple) and the quantum enhanced move;
* $ B\ket{x}_a\ket{y}_b\ket{0}_c = \ket{x}_a\ket{y}_b \left( \sqrt{A_{yx}}\ket{0\cdots 0}_c + \sqrt{1-A_{yx}}\ket{\bot_{xy}}_c \right)$ is the Boltzmann coin, implemented either for the Metropolis-Hasting acceptance or the generalized Glauber acceptance;
* $F\ket{x}_a\ket{y}_b\ket{0\cdots 0}_c = \ket{y}_a\ket{x}_b\ket{0\cdots 0}_c, \; F\ket{x}_a\ket{y}_b\ket{\bot_{xy}}_c = \ket{x}_a\ket{y}_b\ket{\bot_{xy}}_c$ is the accept-path unitary;
* $R_0$ acts as the identity on $\mathcal{H}_a$ and as a reflection about $\ket{0\cdots 0}_b\ket{0\cdots 0}_c$ on $\mathcal{H}_b \otimes \mathcal{H}_c$.


## Reflection circuit

### Multi-controlled NOT

We assume $C \geq 2$ controls. The implementation uses a balanced tree of Toffoli gates:

- first, partial ANDs of the controls are computed into auxiliary qubits;
- then, the resulting root value is used together with the remaining control to flip the target;
- finally, the AND tree is uncomputed, restoring the auxiliary register to $\ket{0\cdots 0}$.

In this resource model, the multi-controlled NOT therefore has:

- number of qubits: $2C-1$ (the implementation uses $C-2$ auxiliary qubits);
- non-Clifford depth upper bound: $2(\log_2(C-1)+1)+1$ (the non-Clifford depth is logarithmic because the AND tree is balanced);
- Toffoli count: $2C-3$ (the tree uses $C-2$ Toffoli gates, the target flip uses one additional Toffoli gate, and the uncomputation uses another $C-2$ Toffoli gates);
- explicit $T$-count: $0$;
- explicit $R_Z$-count: $0$.

In [2]:
from monaqa2.qiskit.multi_controlled_not_symbolic import (mcx_number_qubits, mcx_nc_depth,mcx_t_count,mcx_rz_count,mcx_toffoli_count,)

C = sp.symbols('C', integer=True, positive=True)
display(Markdown(rf"**Resources for a (multi) ${lx(C)}$-controlled NOT**"))
display(Markdown(rf"The number of qubits is: ${lx(mcx_number_qubits(C))}$"))
display(Markdown(rf"The non-Clifford depth is: ${lx(mcx_nc_depth(C))}$"))
display(Markdown(rf"The T-count is: ${lx(mcx_t_count(C))}$"))
display(Markdown(rf"The RZ-count is: ${lx(mcx_rz_count(C))}$"))
display(Markdown(rf"The Toffoli-count is: ${lx(mcx_toffoli_count(C))}$"))

**Resources for a (multi) $C$-controlled NOT**

The number of qubits is: $2 C - 1$

The non-Clifford depth is: $\frac{2 \log{\left(C - 1 \right)}}{\log{\left(2 \right)}} + 3$

The T-count is: $0$

The RZ-count is: $0$

The Toffoli-count is: $2 C - 3$

### Reflection

The reflection operator acts trivially on register $A$ and reflects the joint register $B \otimes C$ around the all-zero state. In other words, the reflected register has size $n+c$, where $n$ is the number of qubits in $B$ and $c$ is the number of coin qubits.

The implementation proceeds as follows:

- apply $X$ gates to all qubits in $B \otimes C$, mapping the all-zero state to the all-one state;
- perform a selective phase flip on the all-one state using a multi-controlled NOT and Hadamards on the target;
- undo the initial $X$ gates.

In [3]:
from monaqa2.qiskit.reflection_symbolic import reflection_number_qubits, reflection_nc_depth, reflection_t_count, reflection_rz_count, reflection_toffoli_count

n, c = sp.symbols('n c', integer=True, positive=True)
display(Markdown(rf"**Resources for a Reflection gate $R_0$ over ${lx(n)}$ spins, ${lx(c)}$ coin qubits**"))
display(Markdown(rf"The number of qubits: ${lx(reflection_number_qubits(n, c))}$"))
display(Markdown(rf"The non-Clifford depth: ${lx(reflection_nc_depth(n, c))}$"))
display(Markdown(rf"The T-count: ${lx(reflection_t_count(n, c))}$"))
display(Markdown(rf"The RZ-count: ${lx(reflection_rz_count(n, c))}$"))
display(Markdown(rf"The Toffoli-count: ${lx(reflection_toffoli_count(n, c))}$"))

**Resources for a Reflection gate $R_0$ over $n$ spins, $c$ coin qubits**

The number of qubits: $2 c + 3 n - 3$

The non-Clifford depth: $\frac{2 \log{\left(c + n - 2 \right)}}{\log{\left(2 \right)}} + 3$

The T-count: $0$

The RZ-count: $0$

The Toffoli-count: $2 c + 2 n - 5$

### Accept path

The accept-path gate implements a controlled swap between registers $A$ and $B$. The swap is applied only when the coin register is in the all-zero state.

The implementation has three main steps:

- first, a multi-controlled NOT checks whether the coin register is $\ket{0\cdots 0}$ and stores the result in a flag qubit;
- then, the flag is fanned out to $n$ control qubits, so that all $n$ controlled swaps between $A$ and $B$ can be applied in parallel;
- finally, the fanout is uncomputed and the flag is reset by applying the same multi-controlled NOT again.

The multi-controlled NOT used to compute the flag has $c$ controls, where $c$ is the number of coin qubits. This operation may itself require auxiliary qubits, as described in the previous section.
The fanout of the flag uses only CNOT gates, so it does not contribute to the non-Clifford resource counts. Its role is to make the $n$ controlled swaps parallel: each bit pair $(A_i,B_i)$ receives its own copy of the flag as control.
Each controlled swap is decomposed into three Toffoli gates. Since the $n$ controlled swaps act on disjoint triples after the fanout, their Toffoli gates can be scheduled in three parallel layers. Therefore the controlled-swap stage contributes $3n$ Toffoli gates and non-Clifford depth $3$.

In [4]:
from monaqa2.qiskit.accept_path_symbolic import (accept_path_number_qubits, accept_path_nc_depth, accept_path_t_count, accept_path_rz_count, accept_path_toffoli_count,)

n, c = sp.symbols('n c', integer=True, positive=True)
display(Markdown(rf"**Resources for Accept-Path over ${lx(n)}$ spins, ${lx(c)}$ coin qubits**"))
display(Markdown(rf"The number of qubits   is: ${lx(accept_path_number_qubits(n, c))}$ (loose upper bound)"))
display(Markdown(rf"The non-Clifford depth is: ${lx(accept_path_nc_depth(n, c))}$"))
display(Markdown(rf"The T-count            is: ${lx(accept_path_t_count(n, c))}$"))
display(Markdown(rf"The RZ-count           is: ${lx(accept_path_rz_count(n, c))}$"))
display(Markdown(rf"The Toffoli-count      is: ${lx(accept_path_toffoli_count(n, c))}$"))

**Resources for Accept-Path over $n$ spins, $c$ coin qubits**

The number of qubits   is: $2 c + 3 n - 2$ (loose upper bound)

The non-Clifford depth is: $\frac{4 \log{\left(c - 1 \right)}}{\log{\left(2 \right)}} + 9$

The T-count            is: $0$

The RZ-count           is: $0$

The Toffoli-count      is: $4 c + 3 n - 6$

## Proposal unitary with uniform move

The uniform proposal acts on two $n$-qubit registers, $A$ and $B$. Register $A$ stores the current configuration, while register $B$ is initialized to $\ket{0^n}$ and used to prepare the proposed configuration.

The implementation applies Hadamard gates to all qubits of register $B$:

- register $A$ is left unchanged;
- register $B$ is mapped from $\ket{0^n}$ to the uniform superposition over all bit strings;

No auxiliary qubits are required.
Since the implementation only uses Hadamard gates, it contains no non-Clifford operations.

In [5]:
from monaqa2.qiskit.proposal_uniform_symbolic import (proposal_uniform_number_qubits, proposal_uniform_nc_depth, proposal_uniform_t_count, proposal_uniform_rz_count, proposal_uniform_toffoli_count)

n = sp.symbols('n', integer=True, positive=True)
display(Markdown(rf"**Resources for Proposal (uniform) on ${lx(n)}$ spins**"))
display(Markdown(rf"The number of qubits   is: ${lx(proposal_uniform_number_qubits(n))}$"))
display(Markdown(rf"The non-Clifford depth is: ${lx(proposal_uniform_nc_depth(n))}$"))
display(Markdown(rf"The T-count            is: ${lx(proposal_uniform_t_count(n))}$"))
display(Markdown(rf"The RZ-count           is: ${lx(proposal_uniform_rz_count(n))}$"))
display(Markdown(rf"The Toffoli-count      is: ${lx(proposal_uniform_toffoli_count(n))}$"))

**Resources for Proposal (uniform) on $n$ spins**

The number of qubits   is: $2 n$

The non-Clifford depth is: $0$

The T-count            is: $0$

The RZ-count           is: $0$

The Toffoli-count      is: $0$

## Proposal unitary with local move

### Dicke state preparation

The Dicke-state preparation circuit prepares the uniform superposition over all $n$-bit strings with Hamming weight $k$. The implementation follows a split-and-merge structure:

- first, the state is initialized in unary form as $\ket{1^k 0^{n-k}}$;
- then, WDB blocks distribute the Hamming weight across a balanced binary partition tree;
- finally, SCS blocks convert the local unary states on the leaves into local Dicke states.

The balanced partition tree is what gives logarithmic depth in $n$, up to the local block costs. Gates acting on disjoint subtrees or leaves can be scheduled in parallel.

The SCS block is the local unary-to-Dicke conversion used on the leaves of the partition tree. For a block with parameter $k$, it acts on $k+1$ qubits.

The implementation consists of one type-I block followed by a sequence of type-II blocks. The type-I block uses a controlled $R_Y$ rotation, while each type-II block uses a doubly controlled $R_Y$ rotation implemented with Toffoli gates and single-qubit rotations.

In [6]:
from monaqa2.qiskit.dicke_preparation_symbolic import scs_number_qubits, scs_nc_depth, scs_t_count, scs_rz_count, scs_toffoli_count

k = sp.symbols("k", integer=True, positive=True)

display(Markdown(rf"**Resources for an SCS block with parameter ${lx(k)}$**"))
display(Markdown(rf"The number of qubits is: ${lx(scs_number_qubits(k))}$"))
display(Markdown(rf"The non-Clifford depth is: ${lx(scs_nc_depth(k))}$"))
display(Markdown(rf"The T-count is: ${lx(scs_t_count(k))}$"))
display(Markdown(rf"The RZ-count is: ${lx(scs_rz_count(k))}$"))
display(Markdown(rf"The Toffoli-count is: ${lx(scs_toffoli_count(k))}$"))

**Resources for an SCS block with parameter $k$**

The number of qubits is: $k + 1$

The non-Clifford depth is: $2 k$

The T-count is: $0$

The RZ-count is: $4 k$

The Toffoli-count is: $2 k - 2$

The WDB block distributes a unary Hamming-weight register across two child blocks of the partition tree. It is the internal-node operation used before the final SCS leaf conversion.

The implementation has three components:

- a CNOT ladder, which changes the unary encoding locally;
- a controlled-addition stage, implemented with controlled and doubly controlled $R_Y$ rotations;
- a Fredkin staircase, which redistributes the weight between the two child registers.

The CNOT ladders are Clifford and do not contribute to the non-Clifford counts. The non-Clifford resources come from the controlled rotations in the addition stage and the Toffoli decomposition of the Fredkin gates.

In [7]:
from monaqa2.qiskit.dicke_preparation_symbolic import wdb_number_qubits, wdb_nc_depth, wdb_t_count, wdb_rz_count, wdb_toffoli_count

n, k = sp.symbols("n k", integer=True, positive=True)

display(Markdown(rf"**Resources for a WDB block on ${lx(n)}$ qubits with parameter ${lx(k)}$**"))
display(Markdown(rf"The number of qubits is: ${lx(wdb_number_qubits(n, k))}$"))
display(Markdown(rf"The non-Clifford depth is: ${lx(wdb_nc_depth(n, k))}$"))
display(Markdown(rf"The T-count is: ${lx(wdb_t_count(n, k))}$"))
display(Markdown(rf"The RZ-count is: ${lx(wdb_rz_count(n, k))}$"))
display(Markdown(rf"The Toffoli-count is: ${lx(wdb_toffoli_count(n, k))}$"))

**Resources for a WDB block on $n$ qubits with parameter $k$**

The number of qubits is: $n$

The non-Clifford depth is: $2 k \left(k + 1\right) + \frac{3 k \left(k + 3\right)}{2} + 2 k$

The T-count is: $0$

The RZ-count is: $k \left(k + 1\right) + 2 k$

The Toffoli-count is: $k \left(k + 1\right) + \frac{3 k \left(k + 3\right)}{2}$

2k^2 + 2k + 1.5k^2 + 4.5k + 2k = 4k^2 + 9k

The full Dicke-state preparation combines WDB and SCS blocks through a balanced partition tree.

Let $L=\lceil n/k\rceil$ be the number of leaves. Each leaf has size at most $k$, and each internal node combines two child blocks using a WDB. Since a binary tree with $L$ leaves has $L-1$ internal nodes, the total gate counts are bounded by the cost of all leaf SCS blocks plus the cost of all internal WDB blocks.

For depth, operations on disjoint subtrees can be parallelized. Therefore the depth scales with the height of the balanced partition tree, rather than with the total number of internal nodes. This gives a logarithmic-in-$L$ contribution from the WDB layers, followed by the local SCS cost on the leaves.

In [8]:
from monaqa2.qiskit.dicke_preparation_symbolic import (dicke_preparation_number_qubits, dicke_preparation_nc_depth, dicke_preparation_t_count, dicke_preparation_rz_count, dicke_preparation_toffoli_count)

n, k = sp.symbols("n k", integer=True, positive=True)

display(Markdown(rf"**Resources for the Dicke state preparation with ${lx(n)}$ qubits and Hamming weight ${lx(k)}$**"))
display(Markdown(rf"Number of qubits: ${lx(dicke_preparation_number_qubits(n, k))}$"))
display(Markdown(rf"Non-Clifford depth: ${lx(dicke_preparation_nc_depth(n, k))}$"))
display(Markdown(rf"Non-Clifford depth for single-spin flip: ${lx(dicke_preparation_nc_depth(n, sp.Integer(1)))}$"))
display(Markdown(rf"T-count: ${lx(dicke_preparation_t_count(n, k))}$"))
display(Markdown(rf"RZ-count: ${lx(dicke_preparation_rz_count(n, k))}$"))
display(Markdown(rf"Toffoli count: ${lx(dicke_preparation_toffoli_count(n, k))}$"))

**Resources for the Dicke state preparation with $n$ qubits and Hamming weight $k$**

Number of qubits: $n$

Non-Clifford depth: $k^{2} + k + \left(\frac{\log{\left(\left\lceil{\frac{n}{k}}\right\rceil \right)}}{\log{\left(2 \right)}} + 1\right) \left(2 k \left(k + 1\right) + \frac{3 k \left(k + 3\right)}{2} + 2 k\right) - 2$

Non-Clifford depth for single-spin flip: $\frac{12 \log{\left(n \right)}}{\log{\left(2 \right)}} + 12$

T-count: $0$

RZ-count: $\left(k \left(k + 1\right) + 2 k\right) \left(\left\lceil{\frac{n}{k}}\right\rceil - 1\right) + \left(2 k^{2} + 2 k - 4\right) \left\lceil{\frac{n}{k}}\right\rceil$

Toffoli count: $\left(k^{2} - k\right) \left\lceil{\frac{n}{k}}\right\rceil + \left(k \left(k + 1\right) + \frac{3 k \left(k + 3\right)}{2}\right) \left(\left\lceil{\frac{n}{k}}\right\rceil - 1\right)$

k^2 + k + (log_2(ceil(n/k)) + 1)(4k^2 + 9k)
log_2(ceil(n/k))(4k^2 + 9k) + 5k^2 + 10k

### Local move proposal

The local proposal uses Dicke-state preparation on register $B$, followed by bitwise CNOTs from $A$ to $B$. The CNOTs map the sampled Hamming-weight-$k$ move string into the proposed configuration $x \oplus z$.

Since the final CNOT layer is Clifford, the proposal has the same non-Clifford depth and gate counts as the Dicke-state preparation. The number of qubits is simply $2n$, corresponding to the two registers $A$ and $B$.

In [9]:
from monaqa2.qiskit.proposal_local_symbolic import (proposal_local_number_qubits, proposal_local_nc_depth, proposal_local_t_count, proposal_local_rz_count, proposal_local_toffoli_count)

n = sp.symbols("n", integer=True, positive=True)
k = sp.Integer(1)  # Use 2, 3, ... for multi-spin flips.

display(Markdown(rf"**Resources for the ${lx(k)}$-spin-flip local proposal**"))
display(Markdown(rf"Number of qubits: ${lx(proposal_local_number_qubits(n, k))}$"))
display(Markdown(rf"Non-Clifford depth: ${lx(proposal_local_nc_depth(n, k))}$"))
display(Markdown(rf"T-count: ${lx(proposal_local_t_count(n, k))}$"))
display(Markdown(rf"RZ-count: ${lx(proposal_local_rz_count(n, k))}$"))
display(Markdown(rf"Toffoli count: ${lx(proposal_local_toffoli_count(n, k))}$"))

**Resources for the $1$-spin-flip local proposal**

Number of qubits: $2 n$

Non-Clifford depth: $\frac{12 \log{\left(n \right)}}{\log{\left(2 \right)}} + 12$

T-count: $0$

RZ-count: $4 n - 4$

Toffoli count: $8 n - 8$

## Proposal unitary with QEMC move

### Controlled-qubitized operator for the transverse-field Ising Hamitonian

The qubitized operator is built from an LCU decomposition of the transverse-field Ising Hamiltonian. The LCU contains the diagonal $Z_i$ and $Z_iZ_j$ Ising terms together with the transverse-field $X_i$ terms.

The uncontrolled qubitized operator consists of the LCU block followed by a reflection around the preparation register. The controlled version adds an external control qubit and controls the prepare, select, and reflection subroutines.

The controlled qubitized operator is the primitive used inside GQSP. Its cost is higher than the uncontrolled qubitized operator because the state-preparation tree and the SELECT operation must be controlled, and because the reflection is also controlled.

In [10]:
from monaqa2.qiskit.qubitized_ising_tf_symbolic import (qubitized_ising_tf_toffoli_count, qubitized_ising_tf_rz_count, qubitized_ising_tf_t_count, qubitized_ising_tf_nc_depth, qubitized_ising_tf_number_qubits, controlled_qubitized_ising_tf_number_qubits, controlled_qubitized_ising_tf_nc_depth, controlled_qubitized_ising_tf_t_count, controlled_qubitized_ising_tf_rz_count, controlled_qubitized_ising_tf_toffoli_count)

n, n_terms = sp.symbols("n m", integer=True, positive=True)

display(Markdown(rf"**Resources for the qubitized transverse-field Ising operator**"))
display(Markdown(rf"Number of qubits: ${lx(qubitized_ising_tf_number_qubits(n, n_terms=n_terms))}$"))
display(Markdown(rf"Non-Clifford depth: ${lx(qubitized_ising_tf_nc_depth(n, n_terms=n_terms))}$"))
display(Markdown(rf"T-count: ${lx(qubitized_ising_tf_t_count(n, n_terms=n_terms))}$"))
display(Markdown(rf"RZ-count: ${lx(qubitized_ising_tf_rz_count(n, n_terms=n_terms))}$"))
display(Markdown(rf"Toffoli count: ${lx(qubitized_ising_tf_toffoli_count(n, n_terms=n_terms))}$"))

display(Markdown(rf"**Resources for the controlled qubitized transverse-field Ising operator**"))
display(Markdown(rf"Number of qubits: ${lx(controlled_qubitized_ising_tf_number_qubits(n, n_terms=n_terms))}$"))
display(Markdown(rf"Non-Clifford depth: ${lx(controlled_qubitized_ising_tf_nc_depth(n, n_terms=n_terms))}$"))
display(Markdown(rf"T-count: ${lx(controlled_qubitized_ising_tf_t_count(n, n_terms=n_terms))}$"))
display(Markdown(rf"RZ-count: ${lx(controlled_qubitized_ising_tf_rz_count(n, n_terms=n_terms))}$"))
display(Markdown(rf"Toffoli count: ${lx(controlled_qubitized_ising_tf_toffoli_count(n, n_terms=n_terms))}$"))

**Resources for the qubitized transverse-field Ising operator**

Number of qubits: $4 m + n - 3$

Non-Clifford depth: $\frac{8 \log{\left(m \right)}}{\log{\left(2 \right)}} + \frac{2 \log{\left(2 m - 3 \right)}}{\log{\left(2 \right)}} + 11$

T-count: $0$

RZ-count: $8 m - 8$

...

Number of qubits: $5 m + 2 n - 1$

Non-Clifford depth: $2 n + \frac{16 \log{\left(m \right)}}{\log{\left(2 \right)}} + \frac{2 \log{\left(2 m - 2 \right)}}{\log{\left(2 \right)}} + 21$

T-count: $0$

RZ-count: $4 m - 4$

Toffoli count: $18 m - 17$

### Hamiltonian simulation via qubitized operator and GQSP

The GQSP Hamiltonian-simulation block approximates $\exp(-iHt)$ using a Jacobi-Anger expansion over the qubitized walk operator. If $\alpha$ is the LCU normalization and $t$ is the simulation time, the relevant scale is $\tau = |\alpha t|$.

The polynomial degree is chosen as $\lceil e|\alpha t|+\log(4/\epsilon)+1\rceil$. The Laurent polynomial is shifted into an ordinary polynomial, which introduces additional powers of the uncontrolled qubitized walk operator.

The resource estimate is therefore expressed in terms of:

- the degree of the Jacobi-Anger approximation;
- the controlled qubitized operator cost, used in the GQSP sequence;
- the uncontrolled qubitized operator cost, used for the Laurent-power shift;
- the single-qubit GQSP phase blocks, which contribute only single-qubit rotations.

In [11]:
from monaqa2.qiskit.hamiltonian_simulation_gqsp_symbolic import (hamiltonian_simulation_gqsp_degree, hamiltonian_simulation_gqsp_number_qubits, hamiltonian_simulation_gqsp_nc_depth, hamiltonian_simulation_gqsp_t_count, hamiltonian_simulation_gqsp_rz_count, hamiltonian_simulation_gqsp_toffoli_count)

n, n_terms, alpha, t, eps = sp.symbols("n m \\alpha t \\varepsilon", integer=True, positive=True)

display(Markdown(rf"**Resources for Hamiltonian simulation via qubitized operator and GQSP**"))
display(Markdown(rf"GQSP degree parameter: $d={lx(hamiltonian_simulation_gqsp_degree(alpha, t, eps))}$"))
display(Markdown(rf"Number of qubits: ${lx(hamiltonian_simulation_gqsp_number_qubits(n, n_terms, alpha, t, eps))}$"))
display(Markdown(rf"Non-Clifford depth: ${lx(hamiltonian_simulation_gqsp_nc_depth(n, n_terms, alpha, t, eps))}$"))
display(Markdown(rf"T-count: ${lx(hamiltonian_simulation_gqsp_t_count(n, n_terms, alpha, t, eps))}$"))
display(Markdown(rf"RZ-count: ${lx(hamiltonian_simulation_gqsp_rz_count(n, n_terms, alpha, t, eps))}$"))
display(Markdown(rf"Toffoli count: ${lx(hamiltonian_simulation_gqsp_toffoli_count(n, n_terms, alpha, t, eps))}$"))

**Resources for Hamiltonian simulation via qubitized operator and GQSP**

GQSP degree parameter: $d=\left\lceil{e \alpha t + \log{\left(\frac{4}{\varepsilon} \right)}}\right\rceil + 1$

Number of qubits: $5 m + 2 n - 1$

Non-Clifford depth: $4 d n + \frac{52 d \log{\left(m \right)}}{\log{\left(2 \right)}} + 59 d + 2 n + \frac{20 \log{\left(m \right)}}{\log{\left(2 \right)}} + 27$

T-count: $0$

RZ-count: $16 m \left\lceil{e \alpha t + \log{\left(\frac{4}{\varepsilon} \right)}}\right\rceil + 20 m - 10 \left\lceil{e \alpha t + \log{\left(\frac{4}{\varepsilon} \right)}}\right\rceil - 8$

Toffoli count: $40 m \left\lceil{e \alpha t + \log{\left(\frac{4}{\varepsilon} \right)}}\right\rceil + 58 m - 41 \left\lceil{e \alpha t + \log{\left(\frac{4}{\varepsilon} \right)}}\right\rceil - 58$

### Hamiltonian simulation via Trotter

As an alternative to GQSP, we also consider a second-order Trotterized implementation of the transverse-field Ising evolution. The Hamiltonian is split as a commuting diagonal part $H_Z$ and a transverse-field part $H_X$.

Each Trotter step uses the Strang formula: a half $X$ layer, a full diagonal $Z/ZZ$ layer, and another half $X$ layer. The $Z_i$ and $X_i$ rotations are single-qubit rotations, while each $Z_iZ_j$ rotation is decomposed using a CNOT--$R_Z$--CNOT pattern.

We do not use $\epsilon$ here. Instead, we fix a constant number of Trotter steps $r$ and report the resource estimate as a function of $r$.

The depth estimate exploits the grouped structure of the implementation. The $X_i$ rotations form a parallel single-qubit layer, the $Z_i$ rotations form another parallel single-qubit layer, and the $Z_iZ_j$ rotations are scheduled by a round-robin edge coloring of the complete graph. Therefore the ZZ part has depth at most $n$ matching rounds, even for dense all-to-all couplings.

Since we fix the number of Trotter steps $r$, we do not use $\epsilon$ in this section. With adjacent $X$ half-layers merged across Strang steps, the grouped non-Clifford depth is bounded by $r(n+2)+1$ when all $X$, $Z$, and $ZZ$ sectors are present.

In [12]:
from monaqa2.qiskit.trotterized_ising_tf_symbolic import (trotterized_ising_tf_number_qubits, trotterized_ising_tf_nc_depth, trotterized_ising_tf_t_count, trotterized_ising_tf_rz_count, trotterized_ising_tf_toffoli_count)

n, n_terms_z, n_terms_zz, n_terms_x, r = sp.symbols("n m_z m_{zz} m_x r", integer=True, positive=True)

display(Markdown(rf"**Resources for Hamiltonian simulation via fixed-step Trotterization**"))
display(Markdown(rf"Number of qubits: ${lx(trotterized_ising_tf_number_qubits(n, num_trotter_steps=r, n_terms_z=n_terms_z, n_terms_zz=n_terms_zz, n_terms_x=n_terms_x))}$"))
display(Markdown(rf"Non-Clifford depth: ${lx(trotterized_ising_tf_nc_depth(n, num_trotter_steps=r, n_terms_z=n_terms_z, n_terms_zz=n_terms_zz, n_terms_x=n_terms_x))}$"))
display(Markdown(rf"T-count: ${lx(trotterized_ising_tf_t_count(n, num_trotter_steps=r, n_terms_z=n_terms_z, n_terms_zz=n_terms_zz, n_terms_x=n_terms_x))}$"))
display(Markdown(rf"RZ-count: ${lx(trotterized_ising_tf_rz_count(n, num_trotter_steps=r, n_terms_z=n_terms_z, n_terms_zz=n_terms_zz, n_terms_x=n_terms_x))}$"))
display(Markdown(rf"Toffoli count: ${lx(trotterized_ising_tf_toffoli_count(n, num_trotter_steps=r, n_terms_z=n_terms_z, n_terms_zz=n_terms_zz, n_terms_x=n_terms_x))}$"))

**Resources for Hamiltonian simulation via fixed-step Trotterization**

Number of qubits: $n$

Non-Clifford depth: $r \left(n + 2\right) + 1$

T-count: $0$

RZ-count: $m_{x} \left(r + 1\right) + r \left(m_{z} + m_{zz}\right)$

Toffoli count: $0$

### Qemc proposal

The QEMC proposal uses Hamiltonian evolution under a transverse-field Ising Hamiltonian to generate a non-local move. The proposal acts on two $n$-qubit registers, $A$ and $B$. Register $A$ stores the current configuration $x$, while register $B$ is initialized to $\ket{0^n}$.

The implementation first copies $A$ into $B$ using bitwise CNOTs. It then applies an approximation of the transverse-field Ising evolution to register $B$. Since the copy layer is Clifford, the non-Clifford resources of the proposal come from the Hamiltonian simulation block.

In [13]:
from monaqa2.qiskit.proposal_qemc_symbolic import (proposal_qemc_number_qubits, proposal_qemc_nc_depth, proposal_qemc_t_count, proposal_qemc_rz_count, proposal_qemc_toffoli_count)

n, n_terms, alpha, t, eps = sp.symbols("n m alpha t \\varepsilon", integer=True, positive=True)

display(Markdown(rf"**Resources for the QEMC proposal with GQSP evolution**"))
display(Markdown(rf"Number of qubits: ${lx(proposal_qemc_number_qubits(n, n_terms, alpha, t, eps))}$"))
display(Markdown(rf"Non-Clifford depth: ${lx(proposal_qemc_nc_depth(n, n_terms, alpha, t, eps))}$"))
display(Markdown(rf"T-count: ${lx(proposal_qemc_t_count(n, n_terms, alpha, t, eps))}$"))
display(Markdown(rf"RZ-count: ${lx(proposal_qemc_rz_count(n, n_terms, alpha, t, eps))}$"))
display(Markdown(rf"Toffoli count: ${lx(proposal_qemc_toffoli_count(n, n_terms, alpha, t, eps))}$"))

**Resources for the QEMC proposal with GQSP evolution**

Number of qubits: $5 m + 3 n - 1$

Non-Clifford depth: $4 d n + \frac{52 d \log{\left(m \right)}}{\log{\left(2 \right)}} + 59 d + 2 n + \frac{20 \log{\left(m \right)}}{\log{\left(2 \right)}} + 27$

T-count: $0$

RZ-count: $16 m \left\lceil{e \alpha t + \log{\left(\frac{4}{\varepsilon} \right)}}\right\rceil + 20 m - 10 \left\lceil{e \alpha t + \log{\left(\frac{4}{\varepsilon} \right)}}\right\rceil - 8$

Toffoli count: $40 m \left\lceil{e \alpha t + \log{\left(\frac{4}{\varepsilon} \right)}}\right\rceil + 58 m - 41 \left\lceil{e \alpha t + \log{\left(\frac{4}{\varepsilon} \right)}}\right\rceil - 58$

## Boltzmann coin with Metropolis-Hastings acceptance

The implementation separates the construction into two parts:

- a reversible fixed-point arithmetic block, which computes a clipped and shifted version of the energy difference;
- a GQSP-based exponential block, which applies the square-root Boltzmann factor to the fixed-point signal.

The reversible fixed-point arithmetic is designed such that the clipped energy difference $\min(0,\Delta E)$ is mapped to the range $[-1, 1)$ (left included, right excluded) with a precision of $b$ bits: one bit for the integer/sign part and $f = b-1$ bits for the fractional part. This format is easier to feed to GQSP later. Note that the largest range we can represent with $f$ fractional bits is $[-1, 1 - 1/2^f]$. 

To map the energy difference to the correct range, we introduce a normalization factor $D = \frac{U}{2 - 1/2^f}$, where $U = 2(\sum_i |h_i| + \sum_{i<j} |J_{ij}|)$ is an upper bound on the absolute energy difference. Then, $\min(0, \Delta E)/D \in [-(2-1/2^f), 0]$, and with a simple shift of $1-1/2^f$ the sum belongs to the desired range. Specifically, the register $q = q_{b-1} \cdots q_0$, which is interpreted as the numerical value $y(q) = -q_{b-1} + \sum_{j=0}^{b-2} 2^{j-f} q_j$ fitting $[-1, 1-1/2^f]$, after the reversible fixed-point block contains 
$$\frac{1}{D}\left(\sum_i h_i x_i^{(A)} + \sum_{i<j} J_{ij} x_i^{(A)} x_j^{(A)} - \sum_i h_i x_i^{(B)} - \sum_{i<j} J_{ij} x_i^{(B)} x_j^{(B)}\right) + \left(1 - \frac{1}{2^f}\right) = \frac{\min(0, \Delta E)}{D} + 1 - \frac{1}{2^f}.$$

This is a sum of $2\left(n + \frac{n(n-1)}{2}\right) + 1$ terms. This can be implemented using a balanced tree of adders, starting an accumulator with $1-1/2^f$ and then performing the $T=2\left(n + \frac{n(n-1)}{2}\right)$ additions. The tree depth is logarithmic in the number of additions to perform, with an additional logarithmic dependence on the word size when Kogge-Stone adders are used.

This register of $b$ bits containing $y = \frac{\min(0, \Delta E)}{D} + 1 - \frac{1}{2^f}$ is processed via GQSP. We need to find a Hamiltonian such that its eigenvalues are the values of $y$. This is 
$$H = - 2^{-b} I + \frac{1}{2} Z_{b-1} - \sum_{j=0}^{b-2} 2^{j-b} Z_j,$$
which can be implemented with an LCU of $b+1$ terms and has one-norm $\alpha = 1$. This Hamiltonian is qubitized, and therefore the scalar function of $y$ is represented as a Laurent polynomial in $z=e^{i\theta}$, where $y=\cos(\theta)=(z+z^{-1})/2$. We therefore target the polynomial
$$P_\text{ideal}(y) = \exp\left( \frac{\beta D}{2} (y-y_\text{max}) \right) = \sqrt{\exp(\beta \min(0, \Delta E))}.$$

This polynomial is inadmissible for synthesis via GQSP because it does not satisfy $|P(y)| \le 1$ for $y \in [-1, 1]$. Indeed, $P_\text{ideal}(1) = \exp(\tau (1 - y_\text{max})) = \exp(\tau 2^{-f}) > 1$, where $\tau=\beta D/2$. It does not matter that our register never allows the value $1$ to appear: the GQSP admissibility condition is imposed on the full signal interval.

We therefore artificially replace $y_\text{max}$ with a new value $\widetilde{y_\text{max}}$. The closest admissible modification is choosing $\widetilde{y_\text{max}} = 1$, but this is numerically fragile in the synthesis because the target touches modulus one. Instead, we choose $\widetilde{y_\text{max}} = 1 + 2^{-f}$. Therefore, we target the polynomial 

$$P_f(y) = \exp\left( \frac{\beta D}{2} (y-\widetilde{y_\text{max}}) \right) = P_\text{ideal}(y) \exp(-\beta D 2^{-f}).$$

The artificial shift makes the target strictly contractive on $[-1,1]$, at the price of a basis-independent multiplicative damping. Since $\exp(-\beta D2^{-f}) = 1 - O(\beta D2^{-f})$, this additional bias becomes negligible when $f$ is large enough.

Allowing a total error $\varepsilon$ in operator norm, we split it as $\varepsilon \ge \varepsilon_\text{fx}(f) + \varepsilon_\text{px}(d)$, where $\varepsilon_\text{fx}(f)$ is the error caused by the truncation to $f$ bits of fractional precision in the fixed-point arithmetic, including the artificial shift, and $\varepsilon_\text{px}(d)$ is the error of the GQSP-based phase arithmetic due to the truncated degree $d$ of the polynomial. 

The error in $f$ is determined by the difference between the ideal and the actual value computed assuming there is no error from truncating the polynomial used to calculate the square root of the exponential. Therefore, the error scales as 
$$\varepsilon_\text{fx}(f) \le \sqrt{T 2^{-f}} \times \exp(\beta D 2^{-f}).$$

For $u = 2^{-f}$ this can be solved via the Lambert-W function:
$$T u e^{2 \beta D u} \le \varepsilon_\text{fx}^2 \Rightarrow u \le \frac{W(2 \beta D \varepsilon_\text{fx}^2 / T)}{2 \beta D}.$$

Since $u=2^{-f}$, this gives
$$f \ge \left\lceil \log_2\left(\frac{2\beta D}{W(2 \beta D \varepsilon_\text{fx}^2 / T)}\right) \right\rceil.$$

The error in $d$ is determined by the truncation of the Laurent polynomial, and we use the same numerical bound used earlier for Hamiltonian simulation:
$$d \ge \left\lceil e \frac{\beta D}{2} + \log(4 \varepsilon_\text{px}^{-1}) + 1 \right\rceil.$$

### Koggle-stone adder

In [14]:
from monaqa2.qiskit.kogge_stone_in_place_adder_symbolic import (kogge_stone_in_place_adder_num_stages, kogge_stone_in_place_adder_prefix_ancillas, kogge_stone_in_place_adder_carry_copy_ancillas, kogge_stone_in_place_adder_number_qubits, kogge_stone_in_place_adder_nc_depth, kogge_stone_in_place_adder_t_count, kogge_stone_in_place_adder_rz_count, kogge_stone_in_place_adder_toffoli_count)

b = sp.Symbol("b", integer=True, positive=True)

display(Markdown(rf"**Resources for the Kogge-Stone in-place adder on ${lx(b)}$-bit registers**"))
display(Markdown(rf"Number of qubits: ${lx(kogge_stone_in_place_adder_number_qubits(b, with_carry_out=False))}$"))
display(Markdown(rf"Non-Clifford depth: ${lx(kogge_stone_in_place_adder_nc_depth(b, with_carry_out=False))}$"))
display(Markdown(rf"T-count: ${lx(kogge_stone_in_place_adder_t_count(b, with_carry_out=False))}$"))
display(Markdown(rf"RZ-count: ${lx(kogge_stone_in_place_adder_rz_count(b, with_carry_out=False))}$"))
display(Markdown(rf"Toffoli count: ${lx(kogge_stone_in_place_adder_toffoli_count(b, with_carry_out=False))}$"))

**Resources for the Kogge-Stone in-place adder on $b$-bit registers**

Number of qubits: $\frac{2 b \log{\left(b \right)}}{\log{\left(2 \right)}} + 3 b + 1$

Non-Clifford depth: $\frac{8 \log{\left(b \right)}}{\log{\left(2 \right)}} + 14$

T-count: $\frac{56 b \log{\left(b \right)}}{\log{\left(2 \right)}} - 28 b + 56$

RZ-count: $0$

Toffoli count: $\frac{8 b \log{\left(b \right)}}{\log{\left(2 \right)}} - 4 b + 8$

### Metropolis-Hastings energy

**Important**: in order to simplify the bounds, we assume $f \ge 10$. 

In [15]:
from monaqa2.qiskit.metropolis_hastings_energy_symbolic import (metropolis_hastings_energy_upper_bound_energy_diff, metropolis_hastings_energy_fractional_bits, metropolis_hastings_energy_acc_word_bits, metropolis_hastings_energy_signal_bits, metropolis_hastings_energy_number_qubits, metropolis_hastings_energy_nc_depth, metropolis_hastings_energy_t_depth, metropolis_hastings_energy_t_count, metropolis_hastings_energy_rz_count, metropolis_hastings_energy_toffoli_count)

n = sp.Symbol("n", integer=True, positive=True)
sum_abs = sp.Symbol(r"U", positive=True)
eps_fx = sp.Symbol(r"\varepsilon_{\mathrm{fx}}", positive=True)
f_ = sp.symbols('f', integer=True, positive=True)

U = metropolis_hastings_energy_upper_bound_energy_diff(sum_abs)
f = metropolis_hastings_energy_fractional_bits(n, sum_abs, eps_fx)
w = metropolis_hastings_energy_acc_word_bits(n, sum_abs, eps_fx)
b = metropolis_hastings_energy_signal_bits(n, sum_abs, eps_fx)
D = sp.simplify(U / (2 - 2 ** (-f_)))

display(Markdown(rf"**Resources for the Metropolis-Hastings fixed-point energy block**"))
display(Markdown(rf"Energy-difference upper bound: ${lx(U)}$"))
display(Markdown(rf"Fractional bits: ${f_}$=${lx(f)}$ without considering shift"))
display(Markdown(rf"Accumulator word bits: ${lx(w)}$"))
display(Markdown(rf"Signal bits: ${lx(b)}$"))
display(Markdown(rf"Normalization: $D$=${lx(D)}$"))
display(Markdown(rf"Number of qubits: ${lx(metropolis_hastings_energy_number_qubits(n, sum_abs, eps_fx))}$"))
display(Markdown(rf"Non-Clifford depth: ${lx(metropolis_hastings_energy_nc_depth(n, sum_abs, eps_fx))}$"))
display(Markdown(rf"T-depth: ${lx(metropolis_hastings_energy_t_depth(n, sum_abs, eps_fx))}$"))
display(Markdown(rf"T-count: ${lx(metropolis_hastings_energy_t_count(n, sum_abs, eps_fx))}$"))
display(Markdown(rf"RZ-count: ${lx(metropolis_hastings_energy_rz_count(n, sum_abs, eps_fx))}$"))
display(Markdown(rf"Toffoli count: ${lx(metropolis_hastings_energy_toffoli_count(n, sum_abs, eps_fx))}$"))

**Resources for the Metropolis-Hastings fixed-point energy block**

Energy-difference upper bound: $2 U$

Fractional bits: $f$=$\frac{3 \log{\left(\frac{2 U n}{\varepsilon_{\mathrm{fx}}} \right)}}{\log{\left(2 \right)}}$ without considering shift

Accumulator word bits: $f + 3$

Signal bits: $f + 1$

...

Non-Clifford depth: $64 \log{\left(f \right)} \log{\left(n \right)} + 48 \log{\left(f \right)} + 72 \log{\left(n \right)} + 62$

T-depth: $0$

T-count: $0$

RZ-count: $0$

Toffoli count: $42 f n^{2} \log{\left(f \right)} + 42 f n \log{\left(f \right)} - 21 f \log{\left(f \right)}$

### Sqrt Exp Arithmetic

In [16]:
from monaqa2.qiskit.sqrt_exp_arithmetic_symbolic import sqrt_exp_arithmetic_degree, sqrt_exp_arithmetic_alpha_signal, sqrt_exp_arithmetic_mu, sqrt_exp_arithmetic_number_qubits, sqrt_exp_arithmetic_nc_depth, sqrt_exp_arithmetic_t_count, sqrt_exp_arithmetic_rz_count, sqrt_exp_arithmetic_toffoli_count

b = sp.Symbol("b", integer=True, positive=True)
beta = sp.Symbol(r"\beta", positive=True)
D = sp.Symbol("D", positive=True)
eps_px = sp.Symbol(r"\varepsilon_{px}", positive=True)

display(Markdown(rf"**Resources for SqrtExpArithmetic**"))
display(Markdown(rf"Signal bits: ${lx(b)}$"))
display(Markdown(rf"Normalization: ${lx(D)}$"))
display(Markdown(rf"Signal normalization: ${lx(sqrt_exp_arithmetic_alpha_signal(b))}$"))
display(Markdown(rf"Polynomial scale: ${lx(sqrt_exp_arithmetic_mu(b, beta, D))}$"))
display(Markdown(rf"Polynomial degree: $d$=${lx(sqrt_exp_arithmetic_degree(b, beta, D, eps_px))}$"))
display(Markdown(rf"Number of qubits: ${lx(sqrt_exp_arithmetic_number_qubits(b, beta, D, eps_px))}$"))
display(Markdown(rf"Non-Clifford depth: ${lx(sqrt_exp_arithmetic_nc_depth(b, beta, D, eps_px))}$"))
display(Markdown(rf"T-count: ${lx(sqrt_exp_arithmetic_t_count(b, beta, D, eps_px))}$"))
display(Markdown(rf"RZ-count: ${lx(sqrt_exp_arithmetic_rz_count(b, beta, D, eps_px))}$"))
display(Markdown(rf"Toffoli count: ${lx(sqrt_exp_arithmetic_toffoli_count(b, beta, D, eps_px))}$"))

**Resources for SqrtExpArithmetic**

Signal bits: $b$

Normalization: $D$

Signal normalization: $1 - 2^{- b}$

Polynomial scale: $\frac{D \beta}{2}$

...

Number of qubits: $7 b + 4$

Non-Clifford depth: $48 d \log{\left(f \right)} + 64 d + 3$

T-count: $0$

RZ-count: $16 b d + 6 d + 3$

Toffoli count: $d \left(38 b - 3\right)$

## Walk operators

### Walk with uniform move

In [17]:
from monaqa2.qiskit.walk_uniform_symbolic import (walk_uniform_number_qubits, walk_uniform_nc_depth, walk_uniform_t_count, walk_uniform_rz_count, walk_uniform_toffoli_count, walk_uniform_coins, walk_uniform_mh_fractional_bits_from_eps, walk_uniform_mh_sqrt_exp_degree_from_eps)
from monaqa2.qiskit.utils_symbolic import get_symbol_name, advanced_initial_simplify, replace_shifted_logs, leading_terms_upper_bound

n = sp.Symbol("n", integer=True, positive=True)
beta = sp.Symbol(r"\beta", positive=True)
sum_abs = sp.Symbol(r"U", positive=True)
eps = sp.Symbol(r"\varepsilon", positive=True)
f = sp.Symbol("f", integer=True, positive=True)
d_px = sp.Symbol(r"d_{px}", integer=True, positive=True)

walk_uniform_qubits_explicit = walk_uniform_number_qubits(n, sum_abs, beta, coin="mh", f=f, d_px=d_px)
walk_uniform_depth_explicit = walk_uniform_nc_depth(n, sum_abs, beta, coin="mh", f=f, d_px=d_px)
walk_uniform_t_count_explicit = walk_uniform_t_count(n, sum_abs, beta, coin="mh", f=f, d_px=d_px)
walk_uniform_rz_count_explicit = walk_uniform_rz_count(n, sum_abs, beta, coin="mh", f=f, d_px=d_px)
walk_uniform_toffoli_count_explicit = walk_uniform_toffoli_count(n, sum_abs, beta, coin="mh", f=f, d_px=d_px)
walk_uniform_coins_explicit = walk_uniform_coins(n, sum_abs, beta, coin="mh", f=f, d_px=d_px)

display(Markdown(rf"**Uniform walk with Metropolis-Hastings coin: explicit formula**"))
display(Markdown(rf"Coin qubits: ${lx(walk_uniform_coins_explicit)}$"))
display(Markdown(rf"Number of qubits: ${lx(walk_uniform_qubits_explicit)}$"))
display(Markdown(rf"Non-Clifford depth: ${lx(walk_uniform_depth_explicit)}$"))
display(Markdown(rf"T-count: ${lx(walk_uniform_t_count_explicit)}$"))
display(Markdown(rf"RZ-count: ${lx(walk_uniform_rz_count_explicit)}$"))
display(Markdown(rf"Toffoli count: ${lx(walk_uniform_toffoli_count_explicit)}$"))

**Uniform walk with Metropolis-Hastings coin: explicit formula**

Coin qubits: $7 f n^{2} \log{\left(f \right)}$

Number of qubits: $22 f n^{2} \log{\left(f \right)}$

Non-Clifford depth: $260 d_{px} \log{\left(f \right)} + 260 \log{\left(f \right)} \log{\left(n \right)} + 6 \log{\left(\log{\left(f \right)} \right)} + 18$

T-count: $1122 f n^{2} \log{\left(f \right)}$

RZ-count: $34 d_{px} f$

Toffoli count: $f \left(77 d_{px} + 202 n^{2} \log{\left(f \right)}\right)$

In [31]:
from monaqa2.qiskit.utils_symbolic import substitute_by_symbol_name, remove_lambertw_upper_bound

eps_fx = eps / 2
eps_px = eps / 2

f_rule = walk_uniform_mh_fractional_bits_from_eps(n, beta, sum_abs/2, eps_fx)
f_rule = remove_lambertw_upper_bound(f_rule).subs(sp.log(2), 1).replace(sp.ceiling, lambda x: x).replace(n+1, 2*n)
d_px_rule = walk_uniform_mh_sqrt_exp_degree_from_eps(beta, sum_abs/2, eps_px)
d_px_rule = d_px_rule.replace(sp.ceiling, lambda x: x)

subs_rules = {
    "f": f_rule,
    "d_{px}": d_px_rule,
}

walk_uniform_qubits_final = substitute_by_symbol_name(walk_uniform_qubits_explicit, subs_rules)
walk_uniform_depth_final = substitute_by_symbol_name(walk_uniform_depth_explicit, subs_rules)
walk_uniform_t_count_final = substitute_by_symbol_name(walk_uniform_t_count_explicit, subs_rules)
walk_uniform_rz_count_final = substitute_by_symbol_name(walk_uniform_rz_count_explicit, subs_rules)
walk_uniform_toffoli_count_final = substitute_by_symbol_name(walk_uniform_toffoli_count_explicit, subs_rules)
walk_uniform_coins_final = substitute_by_symbol_name(walk_uniform_coins_explicit, subs_rules)

display(Markdown(rf"**Uniform walk with Metropolis-Hastings coin: substituted formula**"))
display(Markdown(rf"Fixed-point fractional bits: ${lx(f_rule)}$"))
display(Markdown(rf"Sqrt-exp polynomial degree: ${lx(d_px_rule)}$"))
display(Markdown(rf"Coin qubits: ${lx(walk_uniform_coins_final)}$"))
display(Markdown(rf"Number of qubits: ${lx(walk_uniform_qubits_final)}$"))
display(Markdown(rf"Non-Clifford depth: ${lx(walk_uniform_depth_final.collect([sp.log(sp.log(sum_abs*beta+8*n**2/eps**2))]))}$"))
display(Markdown(rf"T-count: ${lx(walk_uniform_t_count_final)}$"))
display(Markdown(rf"RZ-count: ${lx(walk_uniform_rz_count_final)}$"))
display(Markdown(rf"Toffoli count: ${lx(walk_uniform_toffoli_count_final)}$"))

**Uniform walk with Metropolis-Hastings coin: substituted formula**

Fixed-point fractional bits: $\log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)}$

Sqrt-exp polynomial degree: $\frac{e U \beta}{4} + \log{\left(\frac{8}{\varepsilon} \right)} + 1$

Coin qubits: $7 n^{2} \log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)} \log{\left(\log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)} \right)}$

Number of qubits: $22 n^{2} \log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)} \log{\left(\log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)} \right)}$

Non-Clifford depth: $\left(65 e U \beta + 260 \log{\left(\frac{8}{\varepsilon} \right)} + 260 \log{\left(n \right)} + 260\right) \log{\left(\log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)} \right)} + 6 \log{\left(\log{\left(\log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)} \right)} \right)} + 18$

T-count: $1122 n^{2} \log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)} \log{\left(\log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)} \right)}$

RZ-count: $34 \left(\frac{e U \beta}{4} + \log{\left(\frac{8}{\varepsilon} \right)} + 1\right) \log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)}$

Toffoli count: $\left(\frac{77 e U \beta}{4} + 202 n^{2} \log{\left(\log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)} \right)} + 77 \log{\left(\frac{8}{\varepsilon} \right)} + 77\right) \log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)}$

In [33]:
walk_uniform_depth_final

260*(E*U*\beta/4 + log(8/\varepsilon) + 1)*log(log(U*\beta + 8*n**2/\varepsilon**2)) + 260*log(n)*log(log(U*\beta + 8*n**2/\varepsilon**2)) + 6*log(log(log(U*\beta + 8*n**2/\varepsilon**2))) + 18

### Walk with local move

In [19]:
from monaqa2.qiskit.walk_local_symbolic import (walk_local_number_qubits, walk_local_nc_depth, walk_local_t_count, walk_local_rz_count, walk_local_toffoli_count, walk_local_coins)
from monaqa2.qiskit.walk_uniform_symbolic import (walk_uniform_mh_fractional_bits_from_eps, walk_uniform_mh_sqrt_exp_degree_from_eps)
from monaqa2.qiskit.utils_symbolic import substitute_by_symbol_name, remove_lambertw_upper_bound

In [20]:
n = sp.Symbol("n", integer=True, positive=True)
k = sp.Symbol("k", integer=True, positive=True)
beta = sp.Symbol(r"\beta", positive=True)
sum_abs = sp.Symbol(r"U", positive=True)
eps = sp.Symbol(r"\varepsilon", positive=True)
f = sp.Symbol("f", integer=True, positive=True)
d_px = sp.Symbol(r"d_{px}", integer=True, positive=True)

In [21]:
walk_local_qubits_explicit = walk_local_number_qubits(n, k, sum_abs, beta, coin="mh", f=f, d_px=d_px)
walk_local_depth_explicit = walk_local_nc_depth(n, k, sum_abs, beta, coin="mh", f=f, d_px=d_px)
walk_local_t_count_explicit = walk_local_t_count(n, k, sum_abs, beta, coin="mh", f=f, d_px=d_px)
walk_local_rz_count_explicit = walk_local_rz_count(n, k, sum_abs, beta, coin="mh", f=f, d_px=d_px)
walk_local_toffoli_count_explicit = walk_local_toffoli_count(n, k, sum_abs, beta, coin="mh", f=f, d_px=d_px)
walk_local_coins_explicit = walk_local_coins(n, k, sum_abs, beta, coin="mh", f=f, d_px=d_px)

display(Markdown(rf"**Local walk with Metropolis-Hastings coin: explicit formula**"))
display(Markdown(rf"Coin qubits: ${lx(walk_local_coins_explicit)}$"))
display(Markdown(rf"Number of qubits: ${lx(walk_local_qubits_explicit)}$"))
display(Markdown(rf"Non-Clifford depth: ${lx(walk_local_depth_explicit)}$"))
display(Markdown(rf"T-count: ${lx(walk_local_t_count_explicit)}$"))
display(Markdown(rf"RZ-count: ${lx(walk_local_rz_count_explicit)}$"))
display(Markdown(rf"Toffoli count: ${lx(walk_local_toffoli_count_explicit)}$"))

**Local walk with Metropolis-Hastings coin: explicit formula**

Coin qubits: $7 f n^{2} \log{\left(f \right)}$

Number of qubits: $22 f n^{2} \log{\left(f \right)}$

Non-Clifford depth: $k^{2} \left(24 \log{\left(n \right)} + 70\right) + \left(256 d_{px} + 259 \log{\left(n \right)}\right) \log{\left(f \right)} + 6 \log{\left(\log{\left(f \right)} \right)}$

T-count: $1122 f n^{2} \log{\left(f \right)}$

RZ-count: $34 d_{px} f + 4 k^{2} + 26 k n$

Toffoli count: $77 d_{px} f + 202 f n^{2} \log{\left(f \right)} + 2 k^{2} + 8 k n$

In [22]:
eps_fx = eps / 2
eps_px = eps / 2

f_rule = walk_uniform_mh_fractional_bits_from_eps(n, beta, sum_abs/2, eps_fx)
f_rule = remove_lambertw_upper_bound(f_rule).subs(sp.log(2), 1).replace(sp.ceiling, lambda x: x).replace(n+1, 2*n)

d_px_rule = walk_uniform_mh_sqrt_exp_degree_from_eps(beta, sum_abs/2, eps_px)
d_px_rule = d_px_rule.replace(sp.ceiling, lambda x: x)

subs_rules = {
    "k": sp.Integer(1),
    "f": f_rule,
    "d_{px}": d_px_rule,
}

walk_local_qubits_final = substitute_by_symbol_name(walk_local_qubits_explicit, subs_rules)
walk_local_depth_final = substitute_by_symbol_name(walk_local_depth_explicit, subs_rules)
walk_local_t_count_final = substitute_by_symbol_name(walk_local_t_count_explicit, subs_rules)
walk_local_rz_count_final = substitute_by_symbol_name(walk_local_rz_count_explicit, subs_rules)
walk_local_toffoli_count_final = substitute_by_symbol_name(walk_local_toffoli_count_explicit, subs_rules)
walk_local_coins_final = substitute_by_symbol_name(walk_local_coins_explicit, subs_rules)

display(Markdown(rf"**Local walk with Metropolis-Hastings coin: substituted formula**"))
display(Markdown(rf"Fixed-point fractional bits: ${lx(f_rule)}$"))
display(Markdown(rf"Sqrt-exp polynomial degree: ${lx(d_px_rule)}$"))
display(Markdown(rf"Coin qubits: ${lx(walk_local_coins_final)}$"))
display(Markdown(rf"Number of qubits: ${lx(walk_local_qubits_final)}$"))
display(Markdown(rf"Non-Clifford depth: ${lx(walk_local_depth_final)}$"))
display(Markdown(rf"T-count: ${lx(walk_local_t_count_final)}$"))
display(Markdown(rf"RZ-count: ${lx(walk_local_rz_count_final)}$"))
display(Markdown(rf"Toffoli count: ${lx(walk_local_toffoli_count_final)}$"))

**Local walk with Metropolis-Hastings coin: substituted formula**

Fixed-point fractional bits: $\log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)}$

Sqrt-exp polynomial degree: $\frac{e U \beta}{4} + \log{\left(\frac{8}{\varepsilon} \right)} + 1$

Coin qubits: $7 n^{2} \log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)} \log{\left(\log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)} \right)}$

Number of qubits: $22 n^{2} \log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)} \log{\left(\log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)} \right)}$

Non-Clifford depth: $\left(64 e U \beta + 256 \log{\left(\frac{8}{\varepsilon} \right)} + 259 \log{\left(n \right)} + 256\right) \log{\left(\log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)} \right)} + 24 \log{\left(n \right)} + 6 \log{\left(\log{\left(\log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)} \right)} \right)} + 70$

T-count: $1122 n^{2} \log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)} \log{\left(\log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)} \right)}$

RZ-count: $26 n + 34 \left(\frac{e U \beta}{4} + \log{\left(\frac{8}{\varepsilon} \right)} + 1\right) \log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)} + 4$

Toffoli count: $202 n^{2} \log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)} \log{\left(\log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)} \right)} + 8 n + 77 \left(\frac{e U \beta}{4} + \log{\left(\frac{8}{\varepsilon} \right)} + 1\right) \log{\left(U \beta + \frac{8 n^{2}}{\varepsilon^{2}} \right)} + 2$

### Walk with qemc move

In [26]:
from monaqa2.qiskit.walk_qemc_symbolic import (walk_qemc_number_qubits, walk_qemc_nc_depth, walk_qemc_t_count, walk_qemc_rz_count, walk_qemc_toffoli_count, walk_qemc_coins, walk_qemc_hs_degree_from_eps, walk_qemc_mh_fractional_bits_from_eps, walk_qemc_mh_sqrt_exp_degree_from_eps)
from monaqa2.qiskit.utils_symbolic import substitute_by_symbol_name, remove_lambertw_upper_bound

n = sp.Symbol("n", integer=True, positive=True)
alpha_qemc = sp.Symbol(r"\alpha", positive=True)
t = sp.Symbol("t", positive=True)
beta = sp.Symbol(r"\beta", positive=True)
sum_abs = sp.Symbol(r"U", positive=True)
eps = sp.Symbol(r"\varepsilon", positive=True)

f = sp.Symbol("f", integer=True, positive=True)
d_px = sp.Symbol(r"d_{px}", integer=True, positive=True)
d_hs = sp.Symbol(r"d_{hs}", integer=True, positive=True)

In [29]:
walk_qemc_qubits_explicit = walk_qemc_number_qubits(n, n**2, alpha_qemc, t, sum_abs, beta, coin="mh", f=f, d_px=d_px, d_hs=d_hs)
walk_qemc_depth_explicit = walk_qemc_nc_depth(n, n**2, alpha_qemc, t, sum_abs, beta, coin="mh", f=f, d_px=d_px, d_hs=d_hs)
walk_qemc_t_count_explicit = walk_qemc_t_count(n, n**2, alpha_qemc, t, sum_abs, beta, coin="mh", f=f, d_px=d_px, d_hs=d_hs)
walk_qemc_rz_count_explicit = walk_qemc_rz_count(n, n**2, alpha_qemc, t, sum_abs, beta, coin="mh", f=f, d_px=d_px, d_hs=d_hs)
walk_qemc_toffoli_count_explicit = walk_qemc_toffoli_count(n, n**2, alpha_qemc, t, sum_abs, beta, coin="mh", f=f, d_px=d_px, d_hs=d_hs)
walk_qemc_coins_explicit = walk_qemc_coins(n, n**2, alpha_qemc, t, sum_abs, beta, coin="mh", f=f, d_px=d_px)

display(Markdown(rf"**QEMC walk with GQSP Hamiltonian simulation and Metropolis-Hastings coin: explicit formula**"))
display(Markdown(rf"Coin qubits: ${lx(walk_qemc_coins_explicit)}$"))
display(Markdown(rf"Number of qubits: ${lx(walk_qemc_qubits_explicit)}$"))
display(Markdown(rf"Non-Clifford depth: ${lx(walk_qemc_depth_explicit)}$"))
display(Markdown(rf"T-count: ${lx(walk_qemc_t_count_explicit)}$"))
display(Markdown(rf"RZ-count: ${lx(walk_qemc_rz_count_explicit)}$"))
display(Markdown(rf"Toffoli count: ${lx(walk_qemc_toffoli_count_explicit)}$"))

**QEMC walk with GQSP Hamiltonian simulation and Metropolis-Hastings coin: explicit formula**

Coin qubits: $7 f n^{2} \log{\left(f \right)}$

Number of qubits: $22 f n^{2} \log{\left(f \right)}$

Non-Clifford depth: $118 d_{hs} + 126 d_{px} + n \left(8 d_{hs} + 4\right) + \left(92 d_{px} + 168\right) \log{\left(f \right)} + \left(208 d_{hs} + 128 \log{\left(f \right)} + 440\right) \log{\left(n \right)} + 430$

T-count: $1122 f n^{2} \log{\left(f \right)}$

RZ-count: $d_{hs} d_{px} f + 32 d_{hs} n^{2} + 33 d_{px} f + 8 n^{2}$

Toffoli count: $\frac{201 d_{hs} f n^{2} \log{\left(f \right)}}{20 \log{\left(10 \right)}} + 77 d_{px} f + \frac{469 f n^{2} \log{\left(f \right)}}{8 \log{\left(10 \right)}} + \frac{891 f n^{2} \log{\left(f \right)}}{5}$

In [30]:
eps_fx = eps / 3
eps_px = eps / 3
eps_hs = eps / 3

f_rule = walk_qemc_mh_fractional_bits_from_eps(n, beta, sum_abs/2, eps_fx)
f_rule = remove_lambertw_upper_bound(f_rule).subs(sp.log(2), 1).replace(sp.ceiling, lambda x: x).replace(n+1, 2*n)

d_px_rule = walk_qemc_mh_sqrt_exp_degree_from_eps(beta, sum_abs/2, eps_px)
d_px_rule = d_px_rule.replace(sp.ceiling, lambda x: x)

d_hs_rule = walk_qemc_hs_degree_from_eps(alpha_qemc, t, eps_hs)
d_hs_rule = d_hs_rule.replace(sp.ceiling, lambda x: x)

subs_rules = {
    "f": f_rule,
    "d_{px}": d_px_rule,
    "d_{hs}": d_hs_rule,
}

walk_qemc_qubits_final = substitute_by_symbol_name(walk_qemc_qubits_explicit, subs_rules)
walk_qemc_depth_final = substitute_by_symbol_name(walk_qemc_depth_explicit, subs_rules)
walk_qemc_t_count_final = substitute_by_symbol_name(walk_qemc_t_count_explicit, subs_rules)
walk_qemc_rz_count_final = substitute_by_symbol_name(walk_qemc_rz_count_explicit, subs_rules)
walk_qemc_toffoli_count_final = substitute_by_symbol_name(walk_qemc_toffoli_count_explicit, subs_rules)
walk_qemc_coins_final = substitute_by_symbol_name(walk_qemc_coins_explicit, subs_rules)

display(Markdown(rf"**QEMC walk with GQSP Hamiltonian simulation and Metropolis-Hastings coin: substituted formula**"))
display(Markdown(rf"Fixed-point fractional bits: ${lx(f_rule)}$"))
display(Markdown(rf"Sqrt-exp polynomial degree: ${lx(d_px_rule)}$"))
display(Markdown(rf"Hamiltonian-simulation GQSP degree: ${lx(d_hs_rule)}$"))
display(Markdown(rf"Coin qubits: ${lx(walk_qemc_coins_final)}$"))
display(Markdown(rf"Number of qubits: ${lx(walk_qemc_qubits_final)}$"))
display(Markdown(rf"Non-Clifford depth: ${lx(walk_qemc_depth_final)}$"))
display(Markdown(rf"T-count: ${lx(walk_qemc_t_count_final)}$"))
display(Markdown(rf"RZ-count: ${lx(walk_qemc_rz_count_final)}$"))
display(Markdown(rf"Toffoli count: ${lx(walk_qemc_toffoli_count_final)}$"))

**QEMC walk with GQSP Hamiltonian simulation and Metropolis-Hastings coin: substituted formula**

Fixed-point fractional bits: $\log{\left(U \beta + \frac{18 n^{2}}{\varepsilon^{2}} \right)}$

Sqrt-exp polynomial degree: $\frac{e U \beta}{4} + \log{\left(\frac{12}{\varepsilon} \right)} + 1$

Hamiltonian-simulation GQSP degree: $e \alpha t + \log{\left(\frac{12}{\varepsilon} \right)} + 1$

Coin qubits: $7 n^{2} \log{\left(U \beta + \frac{18 n^{2}}{\varepsilon^{2}} \right)} \log{\left(\log{\left(U \beta + \frac{18 n^{2}}{\varepsilon^{2}} \right)} \right)}$

Number of qubits: $22 n^{2} \log{\left(U \beta + \frac{18 n^{2}}{\varepsilon^{2}} \right)} \log{\left(\log{\left(U \beta + \frac{18 n^{2}}{\varepsilon^{2}} \right)} \right)}$

Non-Clifford depth: $\frac{63 e U \beta}{2} + 118 e \alpha t + n \left(8 e \alpha t + 8 \log{\left(\frac{12}{\varepsilon} \right)} + 12\right) + \left(23 e U \beta + 92 \log{\left(\frac{12}{\varepsilon} \right)} + 260\right) \log{\left(\log{\left(U \beta + \frac{18 n^{2}}{\varepsilon^{2}} \right)} \right)} + \left(208 e \alpha t + 208 \log{\left(\frac{12}{\varepsilon} \right)} + 128 \log{\left(\log{\left(U \beta + \frac{18 n^{2}}{\varepsilon^{2}} \right)} \right)} + 648\right) \log{\left(n \right)} + 244 \log{\left(\frac{12}{\varepsilon} \right)} + 674$

T-count: $1122 n^{2} \log{\left(U \beta + \frac{18 n^{2}}{\varepsilon^{2}} \right)} \log{\left(\log{\left(U \beta + \frac{18 n^{2}}{\varepsilon^{2}} \right)} \right)}$

RZ-count: $32 n^{2} \left(e \alpha t + \log{\left(\frac{12}{\varepsilon} \right)} + 1\right) + 8 n^{2} + \left(\frac{e U \beta}{4} + \log{\left(\frac{12}{\varepsilon} \right)} + 1\right) \left(e \alpha t + \log{\left(\frac{12}{\varepsilon} \right)} + 1\right) \log{\left(U \beta + \frac{18 n^{2}}{\varepsilon^{2}} \right)} + 33 \left(\frac{e U \beta}{4} + \log{\left(\frac{12}{\varepsilon} \right)} + 1\right) \log{\left(U \beta + \frac{18 n^{2}}{\varepsilon^{2}} \right)}$

Toffoli count: $\frac{201 n^{2} \left(e \alpha t + \log{\left(\frac{12}{\varepsilon} \right)} + 1\right) \log{\left(U \beta + \frac{18 n^{2}}{\varepsilon^{2}} \right)} \log{\left(\log{\left(U \beta + \frac{18 n^{2}}{\varepsilon^{2}} \right)} \right)}}{20 \log{\left(10 \right)}} + \frac{469 n^{2} \log{\left(U \beta + \frac{18 n^{2}}{\varepsilon^{2}} \right)} \log{\left(\log{\left(U \beta + \frac{18 n^{2}}{\varepsilon^{2}} \right)} \right)}}{8 \log{\left(10 \right)}} + \frac{891 n^{2} \log{\left(U \beta + \frac{18 n^{2}}{\varepsilon^{2}} \right)} \log{\left(\log{\left(U \beta + \frac{18 n^{2}}{\varepsilon^{2}} \right)} \right)}}{5} + 77 \left(\frac{e U \beta}{4} + \log{\left(\frac{12}{\varepsilon} \right)} + 1\right) \log{\left(U \beta + \frac{18 n^{2}}{\varepsilon^{2}} \right)}$